In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from icechunk_github_actions_demo import Config
from icechunk_github_actions_demo.processing import fetch_annual_lst

In [ ]:
config = Config("config/config_with_secrets_v1.txt")

In [ ]:
tile_row, tile_col = 4, 5
tile_geobox = config.tile_geobox(tile_row, tile_col)
bbox = tile_geobox.boundingbox
print(
    f"Tile ({tile_row}, {tile_col}): "
    f"lat [{bbox.bottom:.1f}, {bbox.top:.1f}], lon [{bbox.left:.1f}, {bbox.right:.1f}]"
)

In [ ]:
import numpy as np
import odc.stac
import pystac_client
import planetary_computer

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"


def fetch_annual_lst_TEST(tile_geobox, year, pixels_per_tile):
    """Fetch MODIS MOD11A2 LST for one tile and one year.

    Returns (avg_lst, max_lst) as uint16 DataArrays, or (None, None) if no data.
    The tile_geobox is passed directly to odc.stac.load to ensure output coordinates
    are pixel-aligned with the global Zarr store.
    """
    odc.stac.configure_rio(cloud_defaults=True)
    bbox = tile_geobox.boundingbox

    catalog = pystac_client.Client.open(STAC_URL,modifier=planetary_computer.sign_inplace)
    items = catalog.search(
        collections=["modis-11A2-061"],
        bbox=[bbox.left, bbox.bottom, bbox.right, bbox.top],
        datetime=f"{year}-01-01/{year}-12-31",
    ).item_collection()

    if not items:
        return None, None

    ds = odc.stac.load(
        items,
        bands=["LST_Day_1km", "QC_Day"],
        geobox=tile_geobox,
        chunks={"time": 1, "x": pixels_per_tile, "y": pixels_per_tile},
    )

    # QC_Day bits 0-1: 0b00 = good, 0b01 = nominal quality
    good = (ds.QC_Day & 0b11) <= 1
    lst = ds.LST_Day_1km.where(good).where(ds.LST_Day_1km >= 7500)

    avg_lst = lst.mean("time").compute().astype(np.uint16)
    max_lst = lst.max("time").compute().astype(np.uint16)
    return avg_lst, max_lst

In [ ]:
bbox = tile_geobox.boundingbox
year=2020
catalog = pystac_client.Client.open(STAC_URL,modifier=planetary_computer.sign_inplace)
search = catalog.search(
    collections=["modis-11A2-061"],
    bbox=[bbox.left, bbox.bottom, bbox.right, bbox.top],
    datetime=f"{year}-01-01/{year}-12-31",
)
search

In [ ]:
items = search.item_collection()
items

In [ ]:
fetch_annual_lst_TEST(tile_geobox, 2020, config.PIXELS_PER_TILE)

In [ ]:
per_year = {}
for year in config.YEARS:
    print(f"  {year}: fetching...")
    avg_lst, max_lst = fetch_annual_lst(tile_geobox, year, config.PIXELS_PER_TILE)
    if avg_lst is None:
        print(f"  {year}: no data, skipping")
        continue
    per_year[year] = {"avg": avg_lst, "max": max_lst}
    print(f"  {year}: done")